In [6]:
import pandas as pd
import numpy as np
import tensorflow as tf

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

df = df.drop(
    columns=["RowNumber", "CustomerId", "Surname"]
)

df["BalanceSalaryRatio"] = (
    df["Balance"] / (df["EstimatedSalary"] + 1)
)

df["AgeBalance"] = (
    df["Age"] * df["Balance"]
)

df["CreditAge"] = (
    df["CreditScore"] * df["Age"]
)

df["ProductsPerTenure"] = (
    df["NumOfProducts"] / (df["Tenure"] + 1)
)

df["ActiveProducts"] = (
    df["IsActiveMember"] * df["NumOfProducts"]
)

df["ZeroBalance"] = (
    df["Balance"] == 0
).astype(int)

X = df.drop("Exited", axis=1)
y = df["Exited"]

categorical_features = [
    "Geography",
    "Gender"
]

numerical_features = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "EstimatedSalary",
    "BalanceSalaryRatio",
    "AgeBalance",
    "CreditAge",
    "ProductsPerTenure",
    "ActiveProducts",
    "ZeroBalance"
]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

configs = [
    ([64, 32], 0.001, 32, 0.1),
    ([128, 64], 0.001, 32, 0.1),
    ([128, 64, 32], 0.001, 32, 0.1),
    ([128, 64, 32], 0.0005, 32, 0.1),
    ([256, 128, 64], 0.0005, 32, 0.1),
    ([256, 128, 64], 0.001, 32, 0.1)
]

best_val_accuracy = 0
best_parameters = None
best_model = None
best_threshold = 0.5

for architecture, learning_rate, batch_size, dropout in configs:

    model = Sequential()

    model.add(
        Input(
            shape=(X_train_processed.shape[1],)
        )
    )

    for units in architecture:

        model.add(
            Dense(
                units,
                activation="relu"
            )
        )

        model.add(
            Dropout(dropout)
        )

    model.add(
        Dense(
            1,
            activation="sigmoid"
        )
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    )

    reduce_lr = ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=0.00001
    )

    history = model.fit(
        X_train_processed,
        y_train,
        validation_data=(
            X_val_processed,
            y_val
        ),
        epochs=80,
        batch_size=batch_size,
        callbacks=[
            early_stopping,
            reduce_lr
        ],
        verbose=0
    )

    val_probability = model.predict(
        X_val_processed,
        verbose=0
    ).ravel()

    current_accuracy = 0
    current_threshold = 0.5

    for threshold in np.arange(0.35, 0.66, 0.01):

        val_prediction = (
            val_probability >= threshold
        ).astype(int)

        val_accuracy = accuracy_score(
            y_val,
            val_prediction
        )

        if val_accuracy > current_accuracy:
            current_accuracy = val_accuracy
            current_threshold = threshold

    print(
        "Architecture:",
        architecture,
        "| Learning Rate:",
        learning_rate,
        "| Batch Size:",
        batch_size,
        "| Dropout:",
        dropout,
        "| Validation Accuracy:",
        round(current_accuracy * 100, 2),
        "%"
    )

    if current_accuracy > best_val_accuracy:

        best_val_accuracy = current_accuracy
        best_model = model
        best_threshold = current_threshold

        best_parameters = {
            "Architecture": architecture,
            "Learning Rate": learning_rate,
            "Batch Size": batch_size,
            "Dropout": dropout,
            "Threshold": round(current_threshold, 2)
        }

train_probability = best_model.predict(
    X_train_processed,
    verbose=0
).ravel()

train_prediction = (
    train_probability >= best_threshold
).astype(int)

train_accuracy = accuracy_score(
    y_train,
    train_prediction
)

test_probability = best_model.predict(
    X_test_processed,
    verbose=0
).ravel()

test_prediction = (
    test_probability >= best_threshold
).astype(int)

test_accuracy = accuracy_score(
    y_test,
    test_prediction
)

print("\n==============================")
print("BEST PARAMETERS")
print("==============================")

print(best_parameters)

print("\n==============================")
print("BEST VALIDATION ACCURACY")
print("==============================")

print(
    round(best_val_accuracy * 100, 2),
    "%"
)

print("\n==============================")
print("TRAINING ACCURACY")
print("==============================")

print(
    round(train_accuracy * 100, 2),
    "%"
)

print("\n==============================")
print("FINAL TEST ACCURACY")
print("==============================")

print(
    round(test_accuracy * 100, 2),
    "%"
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        test_prediction
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        test_prediction
    )
)

print("\nROC-AUC:")

print(
    round(
        roc_auc_score(
            y_test,
            test_probability
        ),
        4
    )
)

Saving Churn_Modelling.csv to Churn_Modelling (5).csv
Architecture: [64, 32] | Learning Rate: 0.001 | Batch Size: 32 | Dropout: 0.1 | Validation Accuracy: 87.33 %
Architecture: [128, 64] | Learning Rate: 0.001 | Batch Size: 32 | Dropout: 0.1 | Validation Accuracy: 87.27 %
Architecture: [128, 64, 32] | Learning Rate: 0.001 | Batch Size: 32 | Dropout: 0.1 | Validation Accuracy: 87.2 %
Architecture: [128, 64, 32] | Learning Rate: 0.0005 | Batch Size: 32 | Dropout: 0.1 | Validation Accuracy: 87.07 %
Architecture: [256, 128, 64] | Learning Rate: 0.0005 | Batch Size: 32 | Dropout: 0.1 | Validation Accuracy: 87.0 %
Architecture: [256, 128, 64] | Learning Rate: 0.001 | Batch Size: 32 | Dropout: 0.1 | Validation Accuracy: 87.0 %

BEST PARAMETERS
{'Architecture': [64, 32], 'Learning Rate': 0.001, 'Batch Size': 32, 'Dropout': 0.1, 'Threshold': np.float64(0.51)}

BEST VALIDATION ACCURACY
87.33 %

TRAINING ACCURACY
86.64 %

FINAL TEST ACCURACY
87.07 %

Classification Report:
              precision